In [6]:
try :
    from datetime import date
    import datetime
    import time
    from catboost import Pool
    from catboost import CatBoostClassifier
    from elasticsearch import Elasticsearch
    from elasticsearch import helpers
    import os
    import sys
    import pandas as pd
    import numpy as np
    import numpy as np

except Exception as e :
    print("Some Modules are Missing{}".format(e))

In [7]:
import elasticsearch
elasticsearch.__version__

(7, 0, 0)

from datetime import date
import datetime
import time
from catboost import Pool
from catboost import CatBoostClassifier
from elasticsearch import Elasticsearch
from elasticsearch import helpers
import os
import sys
import pandas as pd
import numpy as np
import numpy as np

In [8]:
def isOneToOne(df, col1, col2):
    first = df.groupby(col1)[col2].nunique().max()
    second = df.groupby(col2)[col1].nunique().max()
    return first + second == 2
def get_columns(dataset):
    columns = dataset.columns
    res = []
    for i in range (0,len(columns)):
        for j in range (i+1,len(columns)):
            if (isOneToOne(dataset,columns[i],columns[j])):
                res.append((i,j))

                        
    return res 

In [9]:
def generator(df,index_name):
    
    for index,row in df.iterrows():
        mail = False
        if(row["_source.rule.mail"] == 1):
            mail = True
        yield{
            '_index':index_name,
            '_type':'_doc',
            '_id':row.get("_id",' '),
            '_source':{
                'input.type':row.get("_source.input.type","log"),
                'agent.ip':row.get("_source.agent.ip"," "),
                "agent.name":row.get("_source.agent.ip"," "),
                "agent.id":row.get("_source.agent.id"," "),
                "manager.name":row.get("_source.manager.name"," "),
                'data.protocol':row.get('_source.data.protocol'," "),
                "data.srcip":row.get("_source.data.srcip"," "),
                'data.id':row.get('_source.data.id'," "),
                'data.url':row.get('_source.data.url'," "),
                'rule.firedtimes':row.get('_source.rule.firedtimes',1),
                'rule.mail':mail,
                'rule.level':row.get('_source.rule.level',1),
                'rule.description':row.get("_source.rule.description"," "),
                'rule.groups':row.get("_source.rule.groups"," "),
                'rule.id':row.get("_source.rule.id"," "),
                'location':row.get("_source.location"," "),
                'decoder.name':row.get("_source.decoder.name"," "),
                "id":row.get("_source.id"," "),
                "full_log":row.get("_source.full_log"," "),
                "timestamp":row.get("_source.timestamp"," "),
                "rule.pci_dss":row.get("_source.rule.pci_dss"," "),
                "rule.tsc":row.get("_source.rule.tsc"," "),
                "rule.nist_8  _53":row.get("_source.rule.nist_8  _53"," "),
                "rule.gdpr":row.get("_source.rule.gdpr"," "),
                "rule.mitre.id":row.get("_source.rule.mitre.id"," "),
                "rule.hipaa":row.get("_source.rule.hipaa"," "),
                "agent.description":row.get("_source.gent.description"," "),
                "rule.frequency":row.get("_source.rule.frequency"," "),
                "history.rule.firedtimes":row.get('history._source.rule.firedtimes'," "),
                "history.rule.groups":row.get('history._source.rule.groups'," "),
                "history.data.id":row.get('history._source.data.id'," "),
                "history.data.protocol":row.get('history._source.data.protocol'," "),
                "history.count_events":row.get("count_events",0),
                "output":row.get("output"," ")
            }
        }

In [10]:
def preprocessing(df):  

    
    #these category features if they didn't exist we will add them to the dataframe
    list_test_columns= list(df.columns)
    for i in range ( 0, len(category_features)):
        if(category_features[i] not in list_test_columns):
            df[category_features[i]] = " "
    for c in numerical_features:
        if(c not in list_test_columns):
            df[c] = 1
    # fill the columns that has None values with empty strings.
    for c in category_features:
        if(df[c].isnull().any()):
            df[c] = df[c].fillna(' ')
    # fill the columns rule level with rule.level = 3
    df["_source.rule.level"] = df["_source.rule.level"].fillna(1)
    df["_source.rule.firedtimes"] = df["_source.rule.firedtimes"].fillna(1)

    # fill mail values with 1 and 0 
    for index,row in df.iterrows():
        for cat_fe in category_features:
            if(isinstance(row[cat_fe],list)):
                df.at[index,cat_fe] = str(row[cat_fe])
        if (row["_source.rule.mail"]==True):
            df.at[index,"_source.rule.mail"] = 1
        else :
            df.at[index,"_source.rule.mail"] = 0
    df = df.fillna(' ')
    return df

In [11]:
def add_history(df):
    for index,row in df.iterrows():
        src_ip = row["_source.data.srcip"]
        sum_firedtimes = 0
        groups_dict = {}
        id_dict={}
        protocol_dict={}
        num = 0
        for i in range (0,index):
            if(num >= 20):break
            if(df.at[i,"_source.data.srcip"] ==src_ip ):
                num+=1
                sum_firedtimes+= df.at[i,"_source.rule.firedtimes"]

                if(df.at[i,"_source.rule.groups"] in groups_dict.keys()):
                    groups_dict[df.at[i,"_source.rule.groups"]] += 1
                else :
                    groups_dict[df.at[i,"_source.rule.groups"]] = 1

                if(df.at[i,"_source.data.id"] in id_dict.keys()):
                    id_dict[df.at[i,"_source.data.id"]] += 1
                else :
                    id_dict[df.at[i,"_source.data.id"]] = 1

                if(df.at[i,"_source.data.protocol"] in protocol_dict.keys()):
                    protocol_dict[df.at[i,"_source.data.protocol"]] += 1
                else :
                    protocol_dict[df.at[i,"_source.data.protocol"]] = 1
    
    # average of rule firedtimes
        avg_firedtimes = 1
        if(num!= 0):
            avg_firedtimes = sum_firedtimes//num
        df.at[index,"history._source.rule.firedtimes"] = avg_firedtimes

        # the number of history events
        df.at[index,"count_events"] = num

        # the most frequently group
        high_freq_key = " "
        frequency_key = 0
        for key in groups_dict.keys():
            if (groups_dict[key] > frequency_key):
                high_freq_key = key
                frequency_key= groups_dict[key]
        df.at[index,"history._source.rule.groups"] = high_freq_key

        # the most frequenty data.id
        high_freq_key = " "
        frequency_key = 0
        for key in id_dict.keys():
            if (id_dict[key] > frequency_key):
                high_freq_key = key
                frequency_key = id_dict[key]
        df.at[index,"history._source.data.id"] = high_freq_key

        # the most frequent data.protocol
        high_freq_key = " "
        frequency_key = 0
        for key in protocol_dict.keys():
            if (protocol_dict[key] > frequency_key):
                high_freq_key = key
                frequency_key = protocol_dict[key]
        df.at[index,"history._source.data.protocol"] = high_freq_key

    return df
    

In [12]:
def label_data(df):
    model_1 = CatBoostClassifier(
    iterations=1000,
    loss_function='Logloss')
        
    model_2 = CatBoostClassifier(
    iterations=1000,
    loss_function='MultiClass')
    
    model_1.load_model("catboost_model_withContext_1.bin")
    model_2.load_model("catboost_model_withContext_2.bin")
    predictions_1 = model_1.predict(df[all_features])
    df["output"] = predictions_1
    temp_df = df[df["output"] != "NORMAL"]
    predictions_2 = model_2.predict(temp_df[all_features])
    i = 0 
    for index,row in df.iterrows():
        if(row["output"] == "ATTACK"):
            df.at[index,"output"] = predictions_2[i]
            i+=1
    return df["output"]

    

In [13]:

def write_data_to_elasticsearch(df,es):
    
    today = date.today()
    index_name ="wazuh-ai-catboost-"+str(today.year)+'.'+str(today.month)+'.'+str(today.day)
    settings={
        "settings":{
            "number_of_shards":1,
            "number_of_replicas":0,     
        },
        "mappings":{
            "properties":{
                "rule.frequency":{
                    "type":"text"
                },
                "timestamp":{
                    'type':"date",
                    "format": "date_optional_time"
                }
            }
        }
    }
    if(not es.indices.exists(index_name)):
        es.indices.create(index=index_name,body=settings)
    helpers.bulk(es,generator(df,index_name))
    


In [14]:
def get_last_indexed_attack(es):
    last_time_index= es.search(index="wazuh-ai-*",body=
          {
           "size": 1,
           "sort": { "timestamp": "desc"},
           "query": {
              "match_all": {}
           }
        })
    d_f= pd.json_normalize(last_time_index["hits"]["hits"])
    if(d_f.empty):
        last_time = datetime.datetime.now()
        # if there is difference in the time between the agent and the manager 
        last_time = last_time - datetime.timedelta(days=1)
        #'2022-07-13T19:30:22.617+0000'
        last_time = str(last_time.year) + "-" + str(last_time.month) + "-" + str(last_time.day)+ "T" + str(last_time.hour) + ":"+str(last_time.minute) + ":" + str(last_time.second)+".000Z"
    else:
        date_time = d_f["_source.timestamp"][0].split('.')
        last_time = date_time[0] + ".000Z"
        
        
    
    return last_time


In [15]:
query = {
  "version": True,
    "size":1000,
  "sort": [
    {
      "timestamp": {
        "order": "asc",
        "unmapped_type": "boolean"
      }
    }
  ],
  "aggs": {
    "2": {
      "date_histogram": {
        "field": "timestamp",
        "fixed_interval": "30m",
        "time_zone": "Asia/Riyadh",
        "min_doc_count": 1
      }
    }
  },
  "stored_fields": [
    "*"
  ],
  "script_fields": {},
  "docvalue_fields": [
    {
      "field": "data.aws.createdAt",
      "format": "date_time"
    },
    {
      "field": "data.aws.end",
      "format": "date_time"
    },
    {
      "field": "data.aws.resource.instanceDetails.launchTime",
      "format": "date_time"
    },
    {
      "field": "data.aws.service.eventFirstSeen",
      "format": "date_time"
    },
    {
      "field": "data.aws.service.eventLastSeen",
      "format": "date_time"
    },
    {
      "field": "data.aws.start",
      "format": "date_time"
    },
    {
      "field": "data.aws.updatedAt",
      "format": "date_time"
    },
    {
      "field": "data.timestamp",
      "format": "date_time"
    },
    {
      "field": "data.vulnerability.published",
      "format": "date_time"
    },
    {
      "field": "data.vulnerability.updated",
      "format": "date_time"
    },
    {
      "field": "syscheck.mtime_after",
      "format": "date_time"
    },
    {
      "field": "syscheck.mtime_before",
      "format": "date_time"
    },
    {
      "field": "timestamp",
      "format": "date_time"
    }
  ],
  "_source": {
    "excludes": [
      "@timestamp"
    ]
  },
  "query": {
    "bool": {
      "must": [],
      "filter": [
        {
          "match_all": {}
        },
        {
          "range": {
            "timestamp": {
              "gt": "2022-07-12T20:20:02.745Z",
            "format": "date_optional_time"
            }
          }
        }
      ],
      "should": [],
      "must_not": []
    }
  },
  "highlight": {
    "pre_tags": [
      "@kibana-highlighted-field@"
    ],
    "post_tags": [
      "@/kibana-highlighted-field@"
    ],
    "fields": {
      "*": {}
    },
    "fragment_size": 2147483647
  }
}


In [16]:
#2022-07-12T20:20:02.745Z

def run_catboost_ai_multicalssifier_model(es):
    while(True):
        last_index_time= get_last_indexed_attack(es) 
        query["query"]["bool"]["filter"][1]["range"]["timestamp"]["gt"]=last_index_time
        first_qu = es.search(index="wazuh-alerts-*",body=query)
        dataframe = pd.json_normalize(first_qu["hits"]["hits"])
        # get the predictions
        if(not dataframe.empty):
            dataframe = preprocessing(dataframe)
            dataframe = add_history(dataframe)
            dataframe["output"]= label_data(dataframe)
            print(dataframe.shape)
            write_data_to_elasticsearch(dataframe,es)
        time.sleep(10)

        

In [17]:
def __main__():
    user= "wazuh"
    password = "wazuh"
    elasticsearch_ip="192.168.56.103"
    elasticsearch_port = "9200"
    es = Elasticsearch(["https://" + user + ":"+password+"@"+elasticsearch_ip+":"+elasticsearch_port], verify_certs=False)
    if(es.ping()):
            run_catboost_ai_multicalssifier_model(es)
   
    

In [60]:
__main__()

C:\Users\ASUS\anaconda3\lib\site-packages\elasticsearch\connection\http_urllib3.py:204: UserWarning: Connecting to https://192.168.56.103:9200 using SSL with verify_certs=False is insecure.
  warnings.warn(
C:\Users\ASUS\anaconda3\lib\site-packages\urllib3\connectionpool.py:1043: InsecureRequestWarning: Unverified HTTPS request is being made to host '192.168.56.103'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(
C:\Users\ASUS\anaconda3\lib\site-packages\urllib3\connectionpool.py:1043: InsecureRequestWarning: Unverified HTTPS request is being made to host '192.168.56.103'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(
C:\Users\ASUS\anaconda3\lib\site-packages\urllib3\connectionpool.py:1043: InsecureRequestWarning: Unverified HTTPS request is being made to host '192.168.56.103'. Adding

(138, 69)


C:\Users\ASUS\anaconda3\lib\site-packages\elasticsearch\connection\base.py:193: ElasticsearchDeprecationWarning: [types removal] Specifying types in bulk requests is deprecated.
  warnings.warn(message, category=ElasticsearchDeprecationWarning)
C:\Users\ASUS\anaconda3\lib\site-packages\urllib3\connectionpool.py:1043: InsecureRequestWarning: Unverified HTTPS request is being made to host '192.168.56.103'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(
C:\Users\ASUS\anaconda3\lib\site-packages\urllib3\connectionpool.py:1043: InsecureRequestWarning: Unverified HTTPS request is being made to host '192.168.56.103'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(
C:\Users\ASUS\anaconda3\lib\site-packages\elasticsearch\connection\base.py:193: ElasticsearchDeprecationWarning: this request acces

(13, 54)


C:\Users\ASUS\anaconda3\lib\site-packages\urllib3\connectionpool.py:1043: InsecureRequestWarning: Unverified HTTPS request is being made to host '192.168.56.103'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(
C:\Users\ASUS\anaconda3\lib\site-packages\urllib3\connectionpool.py:1043: InsecureRequestWarning: Unverified HTTPS request is being made to host '192.168.56.103'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(
C:\Users\ASUS\anaconda3\lib\site-packages\elasticsearch\connection\base.py:193: ElasticsearchDeprecationWarning: this request accesses system indices: [.kibana_1], but in a future major version, direct access to system indices will be prevented by default
  warnings.warn(message, category=ElasticsearchDeprecationWarning)
C:\Users\ASUS\anaconda3\lib\site-packages\urllib3\con

...

C:\Users\ASUS\anaconda3\lib\site-packages\urllib3\connectionpool.py:1043: InsecureRequestWarning: Unverified HTTPS request is being made to host '192.168.56.103'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(
C:\Users\ASUS\anaconda3\lib\site-packages\urllib3\connectionpool.py:1043: InsecureRequestWarning: Unverified HTTPS request is being made to host '192.168.56.103'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(
C:\Users\ASUS\anaconda3\lib\site-packages\elasticsearch\connection\base.py:193: ElasticsearchDeprecationWarning: this request accesses system indices: [.kibana_1], but in a future major version, direct access to system indices will be prevented by default
  warnings.warn(message, category=ElasticsearchDeprecationWarning)
C:\Users\ASUS\anaconda3\lib\site-packages\urllib3\con

(19, 44)


C:\Users\ASUS\anaconda3\lib\site-packages\urllib3\connectionpool.py:1043: InsecureRequestWarning: Unverified HTTPS request is being made to host '192.168.56.103'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(
C:\Users\ASUS\anaconda3\lib\site-packages\urllib3\connectionpool.py:1043: InsecureRequestWarning: Unverified HTTPS request is being made to host '192.168.56.103'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(
C:\Users\ASUS\anaconda3\lib\site-packages\elasticsearch\connection\base.py:193: ElasticsearchDeprecationWarning: this request accesses system indices: [.kibana_1], but in a future major version, direct access to system indices will be prevented by default
  warnings.warn(message, category=ElasticsearchDeprecationWarning)
C:\Users\ASUS\anaconda3\lib\site-packages\urllib3\con

(12, 44)


KeyboardInterrupt: 

In [21]:

user= "wazuh"
password = "wazuh"

elasticsearch_ip="192.168.56.104"
elasticsearch_port = "9200"
es = Elasticsearch(["https://" + user + ":"+password+"@"+elasticsearch_ip+":"+elasticsearch_port], verify_certs=False)
es.ping()

False